# Prepare Base Model Stage

This notebook mirrors the second stage of the project pipeline: load the project parameters, build the VGG16 base model, and create the updated classification head.

In [ ]:
import tensorflow as tf
from pathlib import Path
import yaml

with open("params.yaml", "r", encoding="utf-8") as file_obj:
    params = yaml.safe_load(file_obj)

params

In [ ]:
base_model = tf.keras.applications.VGG16(
    input_shape=tuple(params["IMAGE_SIZE"]),
    weights=params["WEIGHTS"],
    include_top=params["INCLUDE_TOP"]
)
base_model.summary()

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

flatten_in = tf.keras.layers.Flatten(name="flatten")(base_model.output)
prediction = tf.keras.layers.Dense(params["CLASSES"], activation="softmax", name="prediction")(flatten_in)

full_model = tf.keras.Model(inputs=base_model.input, outputs=prediction)
full_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=params["LEARNING_RATE"]),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)
full_model.summary()

In [ ]:
output_dir = Path("artifacts/prepare_base_model")
output_dir.mkdir(parents=True, exist_ok=True)
base_model.save(output_dir / "base_model.h5")
full_model.save(output_dir / "base_model_updated.h5")
print("Base model artifacts saved")